In [2]:
import os
import nd2
import h5py
import numpy as np
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display

def convert_nd2_to_hdf5(nd2_path, h5_path, dataset_name="frames", normalize_to_8bit=True, lut_min=None, lut_max=None, start_frame=0, end_frame=None):
    if not os.path.exists(nd2_path):
        raise FileNotFoundError(f"Cannot find the ND2 file: {nd2_path}")

    print(f"\nOpening {nd2_path}...")
    
    with nd2.ND2File(nd2_path) as f:
        print(f"Original ND2 shape: {f.shape}")
        print(f"ND2 dtype: {f.dtype}")
        
        # MAC OPTIMIZATION: Always use lazy loading via dask to protect system memory
        video_data = f.to_dask()
        
        # Safely extract core spatial/temporal shapes
        total_frames = f.sizes.get('T', f.shape[0])
        height = f.sizes.get('Y', f.shape[-2])
        width = f.sizes.get('X', f.shape[-1])
        
        start_frame = max(0, start_frame)
        end_frame = total_frames if end_frame is None else min(end_frame, total_frames)
        target_num_frames = end_frame - start_frame
        
        if target_num_frames <= 0:
            raise ValueError(f"Invalid frame range: Start ({start_frame}) must be less than End ({end_frame}).")
            
        print(f"Target HDF5 shape: ({target_num_frames}, {height}, {width})  [Frames {start_frame} to {end_frame-1}]")

        used_min, used_max = 0, 255
        
        # --- Helper to force any weird frame shape down to an explicit 2D slice ---
        def extract_2d_frame(raw_data):
            sq_data = np.squeeze(raw_data)
            while len(sq_data.shape) > 2:
                sq_data = sq_data[0] # Grab the first channel/z-stack if multi-dimensional
            return sq_data.compute() if hasattr(sq_data, 'compute') else sq_data

        # --- High-Efficiency Contrast LUT Estimation ---
        if normalize_to_8bit:
            if lut_min is None or lut_max is None:
                print(f"Calculating global min/max across target range without loading full video to RAM...")
                g_min = float('inf')
                g_max = float('-inf')
                
                # Mac Safety Check: Sample up to 10 spaced frames rather than looping through thousands
                sample_step = max(1, target_num_frames // 10)
                for idx in range(start_frame, end_frame, sample_step):
                    frame_sample = extract_2d_frame(video_data[idx])
                    g_min = min(g_min, frame_sample.min())
                    g_max = max(g_max, frame_sample.max())
                
                used_min = lut_min if lut_min is not None else g_min
                used_max = lut_max if lut_max is not None else g_max
            else:
                used_min = lut_min
                used_max = lut_max
            
            print(f"LUT Parameters Applied -> Min: {used_min} | Max: {used_max}")

        # --- Streamlined Safe HDF5 Generation Block ---
        with h5py.File(h5_path, 'w') as h5f:
            target_dtype = np.uint8 if normalize_to_8bit else f.dtype
            
            dataset = h5f.create_dataset(
                dataset_name, 
                shape=(target_num_frames, height, width), 
                dtype=target_dtype,
                chunks=(1, height, width), # Chunked per frame for rapid streaming reads
                compression="gzip", 
                compression_opts=4
            )
            
            print(f"Writing chunks sequentially to {h5_path}...")
            out_idx = 0
            
            # Use progress bar to stream chunks one-by-one safely down onto disk storage
            for i in tqdm(range(start_frame, end_frame), desc="Converting Frames"):
                frame = extract_2d_frame(video_data[i])
                
                if normalize_to_8bit and frame.dtype != np.uint8:
                    frame_float = frame.astype(np.float32)
                    frame_float = np.clip(frame_float, used_min, used_max)
                    
                    if used_max > used_min:
                        frame = ((frame_float - used_min) / (used_max - used_min)) * 255.0
                    else:
                        frame = frame_float - used_min
                    frame = frame.astype(np.uint8)
                
                dataset[out_idx, :, :] = frame
                out_idx += 1

    print("Conversion complete!")
    if normalize_to_8bit:
        print(f"--- FINAL LUT SAVED -> Min: {used_min} | Max: {used_max} ---\n")

In [ ]:
import os
import re
import csv
import nd2              # Added missing library import for handling .nd2 files
import numpy as np
import pandas as pd
import tkinter as tk
import matplotlib.pyplot as plt
import ipywidgets as widgets
from tkinter import filedialog
from IPython.display import display, clear_output

# --- Paste your exact file path here ---
nd2_path = r"/Users/dharani/Desktop/data/20260324_1_1.nd2" 

def interactive_lut_preview(nd2_path):
    if not os.path.exists(nd2_path):
        print(f"Error: File not found at {nd2_path}")
        return
        
    print(f"Loading preview for: {nd2_path}")
    
    f = nd2.ND2File(nd2_path)
    if 'T' in f.sizes:
        total_frames = f.sizes['T']
    else:
        total_frames = f.shape[0] if len(f.shape) > 2 else 1
        
    video_data = f.to_dask() 
    
    def extract_2d_frame(raw_data):
        sq_data = np.squeeze(raw_data)
        while len(sq_data.shape) > 2:
            sq_data = sq_data[0]
        return sq_data.compute() if hasattr(sq_data, 'compute') else sq_data

    first_frame = extract_2d_frame(video_data[0])
    
    max_slider_val = int(first_frame.max() * 1.5) 
    if max_slider_val <= 0: 
        max_slider_val = 65535 
        
    # --- VS Code specific interactive display setup ---
    output_window = widgets.Output()  # Creates a dedicated UI container for the plot

    def update_view(frame_idx, lut_min, lut_max):
        raw_frame = video_data[frame_idx] if len(video_data.shape) > 2 else video_data
        frame = extract_2d_frame(raw_frame)
        
        frame_float = frame.astype(np.float32)
        frame_float = np.clip(frame_float, lut_min, lut_max)
        
        if lut_max > lut_min:
            frame_norm = ((frame_float - lut_min) / (lut_max - lut_min)) * 255.0
        else:
            frame_norm = frame_float - lut_min
            
        frame_uint8 = np.uint8(frame_norm)
        
        # Use the container to update the rendering without locking the VS Code thread
        with output_window:
            clear_output(wait=True)  # Instantly wipe the old image
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.imshow(frame_uint8, cmap='gray', vmin=0, vmax=255)
            ax.set_title(f"Frame {frame_idx} | Min: {lut_min} | Max: {lut_max}")
            ax.axis('off')
            plt.show()               # Explicitly show the fresh render
            plt.close(fig)           # Immediately close to prevent Mac memory leaks

    # 5. Interactive sliders
    slider_ui = widgets.interactive(
        update_view,
        frame_idx=widgets.IntSlider(min=0, max=total_frames-1, step=1, value=0, description='Frame:', continuous_update=False),
        lut_min=widgets.IntSlider(min=0, max=max_slider_val, step=1, value=int(first_frame.min()), description='LUT Min:', continuous_update=False),
        lut_max=widgets.IntSlider(min=1, max=max_slider_val, step=1, value=int(first_frame.max()), description='LUT Max:', continuous_update=False)
    )
    
    # Render both the controls and our updated output window explicitly
    display(slider_ui, output_window)

# Run the viewer
interactive_lut_preview(nd2_path)

Loading preview for: /Users/dharani/Desktop/data/20260324_1_1.nd2


interactive(children=(IntSlider(value=0, continuous_update=False, description='Frame:', max=1363), IntSlider(v…

Output()

In [5]:
# Set up your file targets manually 
input_nd2 = r"/Users/dharani/Desktop/data/20260324_1_1.nd2"
output_h5 = r"/Users/dharani/Desktop/data/20260324_1_1.h5"

# Run it!
convert_nd2_to_hdf5(
    nd2_path=input_nd2,
    h5_path=output_h5,
    normalize_to_8bit=True,
    lut_min=97,
    lut_max=123,
    start_frame=0, 
    end_frame=None        
)


Opening /Users/dharani/Desktop/data/20260324_1_1.nd2...
Original ND2 shape: (1364, 510, 522)
ND2 dtype: uint16
Target HDF5 shape: (1364, 510, 522)  [Frames 0 to 1363]
LUT Parameters Applied -> Min: 97 | Max: 123
Writing chunks sequentially to /Users/dharani/Desktop/data/20260324_1_1.h5...


Converting Frames: 100%|██████████| 1364/1364 [00:19<00:00, 69.20it/s]


Conversion complete!
--- FINAL LUT SAVED -> Min: 97 | Max: 123 ---

